# Uber Data Engineering Project

Cleaned and Improved Version

This notebook builds a proper star schema from Uber trip data with optimized joins and deduplicated dimensions.

## 1. Import Libraries

In [ ]:

import pandas as pd


## 2. Load Dataset

In [ ]:

url = "https://storage.googleapis.com/shubham-de-project-uber/uber_data.csv"
df = pd.read_csv(url)
df.head()


## 3. Data Cleaning & Preprocessing

In [ ]:

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df = df.drop_duplicates().reset_index(drop=True)
df['trip_id'] = df.index
df.info()


## 4. Datetime Dimension

In [ ]:

datetime_dim = df[['tpep_pickup_datetime']].drop_duplicates().reset_index(drop=True)

datetime_dim['datetime_id'] = datetime_dim.index
datetime_dim['hour'] = datetime_dim['tpep_pickup_datetime'].dt.hour
datetime_dim['day'] = datetime_dim['tpep_pickup_datetime'].dt.day
datetime_dim['month'] = datetime_dim['tpep_pickup_datetime'].dt.month
datetime_dim['year'] = datetime_dim['tpep_pickup_datetime'].dt.year
datetime_dim['weekday'] = datetime_dim['tpep_pickup_datetime'].dt.weekday

datetime_dim.head()


## 5. Passenger Count Dimension

In [ ]:

passenger_count_dim = df[['passenger_count']].drop_duplicates().reset_index(drop=True)
passenger_count_dim['passenger_count_id'] = passenger_count_dim.index
passenger_count_dim.head()


## 6. Trip Distance Dimension

In [ ]:

trip_distance_dim = df[['trip_distance']].drop_duplicates().reset_index(drop=True)
trip_distance_dim['trip_distance_id'] = trip_distance_dim.index
trip_distance_dim.head()


## 7. Rate Code Dimension

In [ ]:

rate_code_dim = df[['RatecodeID']].drop_duplicates().reset_index(drop=True)
rate_code_dim['rate_code_id'] = rate_code_dim.index
rate_code_dim.head()


## 8. Payment Type Dimension

In [ ]:

payment_type_dim = df[['payment_type']].drop_duplicates().reset_index(drop=True)
payment_type_dim['payment_type_id'] = payment_type_dim.index
payment_type_dim.head()


## 9. Location Dimensions

In [ ]:

pickup_location_dim = df[['pickup_latitude', 'pickup_longitude']]\
    .drop_duplicates().reset_index(drop=True)
pickup_location_dim['pickup_location_id'] = pickup_location_dim.index

dropoff_location_dim = df[['dropoff_latitude', 'dropoff_longitude']]\
    .drop_duplicates().reset_index(drop=True)
dropoff_location_dim['dropoff_location_id'] = dropoff_location_dim.index


## 10. Fact Table

In [ ]:

fact_table = df \
.merge(passenger_count_dim, on='passenger_count') \
.merge(trip_distance_dim, on='trip_distance') \
.merge(rate_code_dim, on='RatecodeID') \
.merge(payment_type_dim, on='payment_type') \
.merge(pickup_location_dim, on=['pickup_latitude','pickup_longitude']) \
.merge(dropoff_location_dim, on=['dropoff_latitude','dropoff_longitude']) \
.merge(datetime_dim, on='tpep_pickup_datetime')

fact_table = fact_table[[
    'trip_id',
    'datetime_id',
    'passenger_count_id',
    'trip_distance_id',
    'rate_code_id',
    'payment_type_id',
    'pickup_location_id',
    'dropoff_location_id',
    'fare_amount',
    'total_amount'
]]

fact_table.head()


## 11. Summary


- Built clean star schema  
- Deduplicated all dimensions  
- Used business-key joins  
- Interview-ready data model  
